In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.interpolate import interp1d
from scipy.integrate import quad, cumulative_trapezoid

In [3]:
energies = np.logspace(0, 4, 100)[0:1]

In [4]:
survival = np.zeros((100, 10))
std_dev = np.zeros((100, 10))
avg_dev = np.zeros((100, 10))
for i in range(len(energies)):
    filepath = f"build/output/muon_track_{i}.txt"
    df = pd.read_csv(filepath, sep=r'\s+', header=0, usecols=[1, 2, 4], 
                    names=['x', 'y', 'rec_e'], dtype={'x': float, 'y': float, 'rec_e': float})
    x_positions = df['x'].values.reshape(-1, 10)
    y_positions = df['y'].values.reshape(-1, 10)
    rec_energies = df['rec_e'].values.reshape(-1, 10)
    dev = np.sqrt(x_positions**2 + y_positions**2)

    sums = dev.sum(axis=0)

    counts = (dev != 0).sum(axis=0)

    survival[i] = counts / 1000

    avg  = np.divide(sums, counts, out=np.zeros_like(sums, dtype=float), where=counts!=0)

    avg_dev[i] = avg

    std_dev[i] = np.sqrt(np.divide(((dev - avg)**2).sum(axis=0), counts, out=np.zeros_like(sums, dtype=float), where=counts!=0))

In [9]:
size_m2 = 10
int_time = 600

In [10]:
e, flux = np.loadtxt("H3a.txt", usecols=(0, 1), unpack=True)
flux_interp = interp1d(e, flux, kind='linear', fill_value="extrapolate")
integral_flux, _ = quad(flux_interp, 1, 10000)

C:\Users\Claudio\AppData\Local\Temp\ipykernel_18012\2209594210.py:3: IntegrationWarning: The maximum number of subdivisions (50) has been achieved.
  If increasing the limit yields no improvement it is advised to analyze 
  the integrand in order to determine the difficulties.  If the position of a 
  local difficulty can be determined (singularity, discontinuity) one will 
  probably gain from splitting up the interval and calling the integrator 
  on the subranges.  Perhaps a special-purpose integrator should be used.
  integral_flux, _ = quad(flux_interp, 1, 10000)


In [11]:
flux_in_scenario = integral_flux * size_m2 * int_time * np.pi * 4

In [12]:
e_grid = np.logspace(0, 4, 100)

pdf = flux_interp(e_grid)
pdf = np.maximum(pdf, 0)

cdf = cumulative_trapezoid(pdf, e_grid, initial=0)

cdf_normalized = cdf / cdf[-1]

inv_cdf = interp1d(cdf_normalized, e_grid, bounds_error=False, fill_value=(e_grid[0], e_grid[-1]))

random_u = np.random.uniform(0, 1, int(flux_in_scenario))

sampled_energies = inv_cdf(random_u)

In [14]:
for ee in sampled_energies:

    ei = np.digitize(ee, energies)
    for l in range(10):
        survived = np.random.rand() < survival[ei, l]
        if survived:
            dev_value = np.random.normal(avg_dev[ei, l], std_dev[ei, l])
        else:
            break